In [1]:
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
import re
import numpy as np

#from sklearn.preprocessing import OneHotEncoder

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train_df = pd.read_csv("df.csv")
test_df = pd.read_csv("df_test.csv")

In [3]:
# 東京都23区のリスト
wards = [
    "千代田区", "中央区", "港区", "新宿区", "文京区", "台東区", "墨田区", "江東区",
    "品川区", "目黒区", "大田区", "世田谷区", "渋谷区", "中野区", "杉並区", "豊島区",
    "北区", "荒川区", "板橋区", "練馬区", "足立区", "葛飾区", "江戸川区"
]

In [4]:
# 正規表現を使って「何区」を抽出する関数
def extract_ward(address):
    # 23区リストを元に住所から該当する区を検索
    for ward in wards:
        if ward in address:
            return ward
    return None  # 該当する区がなければ None を返す

In [5]:
# 住所データを含む列に対して処理を適用
train['所在地'] = train['所在地'].apply(extract_ward)
test['所在地'] = test['所在地'].apply(extract_ward)

In [6]:
# 所在地を数値に変換するマッピング
#location_mapping = {location: idx for idx, location in enumerate(train['所在地'].unique())}

In [7]:
# 所在地を数値に変換
#train['所在地'] = train['所在地'].map(location_mapping)
#test['所在地'] = test['所在地'].map(location_mapping)

In [8]:
# 所在地のマッピングを表示
#for location, idx in location_mapping.items():
#    print(f"{location}: {idx}")

In [9]:
# 別で作成した緯度と経度を持ってくる
train['緯度'] = train_df['緯度']
train['経度'] = train_df['経度']
test['緯度'] = test_df['緯度']
test['経度'] = test_df['経度']

In [10]:
# 区ごとの緯度・経度の平均を計算
mean_lat_long = train.groupby('所在地')[['緯度', '経度']].mean()

In [11]:
# 欠損値を区ごとの平均値で補完する関数
def fill_missing_lat_long(row):
    if pd.isnull(row['緯度']) or pd.isnull(row['経度']):
        # 区ごとの平均値を取得
        mean_values = mean_lat_long.loc[row['所在地']]
        if pd.isnull(row['緯度']):
            row['緯度'] = mean_values['緯度']
        if pd.isnull(row['経度']):
            row['経度'] = mean_values['経度']
    return row

In [12]:
# 欠損値を補完
train = train.apply(fill_missing_lat_long, axis=1)
test = test.apply(fill_missing_lat_long, axis=1)

In [13]:
# 徒歩時間を正規表現で抽出し、最短のものを取得する関数
def extract_min_walk_time(access_str):
    # 正規表現で「徒歩○分」を抽出
    walk_times = re.findall(r'徒歩(\d+)分', access_str)
    # 数字に変換して最小値を返す
    if walk_times:
        return min(map(int, walk_times))
    return None

In [14]:
# アクセスを最短時間のみに
train['アクセス'] = train['アクセス'].apply(extract_min_walk_time)
test['アクセス'] = test['アクセス'].apply(extract_min_walk_time)

In [15]:
# アクセスを範囲で変換する関数
def categorize_access(access):
    if access < 10:
        return 1
    elif access < 20:
        return 2
    else:
        return 3

In [16]:
# アクセス列を変換
train['アクセス'] = train['アクセス'].apply(categorize_access)
test['アクセス'] = test['アクセス'].apply(categorize_access)

In [17]:
# 間取りの'+S(納戸)'を除去
train['間取り'] = train['間取り'].str.replace(r'\+S\(納戸\)', '', regex=True)
test['間取り'] = test['間取り'].str.replace(r'\+S\(納戸\)', '', regex=True)

In [18]:
# 間取りを数値に変換するマッピング
location_mapping2 = {location: idx for idx, location in enumerate(train['間取り'].unique())}

In [19]:
# 間取りを数値に変換
train['間取り'] = train['間取り'].map(location_mapping2)
test['間取り'] = test['間取り'].map(location_mapping2)

In [20]:
# 間取りのマッピングを表示
for location, idx in location_mapping2.items():
    print(f"{location}: {idx}")

1K: 0
1R: 1
2LDK: 2
2DK: 3
1DK: 4
1LDK: 5
3LDK: 6
3DK: 7
4K: 8
2K: 9
4LDK: 10
5LDK: 11
3K: 12
4DK: 13
5DK: 14
5K: 15
6LDK: 16
1LK: 17


In [21]:
# 築年数を数値に変換する関数
def convert_age(age_str):
    # 正規表現で「年」と「ヶ月」を分離
    match = re.match(r'(\d+)年(\d+)ヶ月', age_str)
    if match:
        years = int(match.group(1))  # 年
        months = int(match.group(2))  # ヶ月
        # 年とヶ月を数値に変換 (1年 = 12ヶ月)
        return years + months / 12
    else:
        # ○年だけの場合
        match = re.match(r'(\d+)年', age_str)
        if match:
            years = int(match.group(1))
            return years
        # ○ヶ月だけの場合
        match = re.match(r'(\d+)ヶ月', age_str)
        if match:
            months = int(match.group(1))
            return months / 12
        return None  # 不正なデータはNoneとする

In [22]:
# 築年数列を変換
train['築年数'] = train['築年数'].apply(convert_age)
test['築年数'] = test['築年数'].apply(convert_age)

In [23]:
# NaNを0に置き換え
train['築年数'] = train['築年数'].fillna(0).astype(int)
test['築年数'] = test['築年数'].fillna(0).astype(int)

In [24]:
# 50年以上のデータを削除
#train = train[train['築年数'] < 50]
#test = test[test['築年数'] < 50]

In [25]:
# 築年数を範囲で変換する関数
def categorize_access(yconst):
    if yconst < 10:
        return 1
    elif yconst < 20:
        return 2
    elif yconst < 30:
        return 3
    elif yconst < 40:
        return 4
    elif yconst < 50:
        return 5
    else:
        return 6

In [26]:
# 築年数列を変換
train['築年数'] = train['築年数'].apply(categorize_access)
test['築年数'] = test['築年数'].apply(categorize_access)

# 方角列の欠損を削除
train = train.dropna(subset=['方角'])

# テストデータは削れないからブランクに無と入力
test['方角'] = test['方角'].replace('', np.nan)  # 空文字列をNaNに置換
test['方角'] = test['方角'].fillna('無')         # NaNを「無」に置換

# 南フラグ列の追加
train['南フラグ'] = train['方角'].apply(lambda x: 1 if '南' in x else 0)
test['南フラグ'] = test['方角'].apply(lambda x: 1 if '南' in x else 0)

In [27]:
# 正規表現で数値部分のみを抽出
#train['面積'] = test['面積'].str.extract(r'(\d+\.\d+|\d+)').astype(float)
#test['面積'] = test['面積'].str.extract(r'(\d+\.\d+|\d+)').astype(float)

In [28]:
# m2を削除して数値に変換
train['面積'] = train['面積'].str.replace('m2', '').astype(float)
test['面積'] = test['面積'].str.replace('m2', '').astype(float)

In [29]:
# 面積を範囲で変換する関数
def categorize_area(area):
    if area < 25:
        return 1
    elif area < 50:
        return 2
    elif area < 75:
        return 3
    elif area < 100:
        return 4
    else:
        return 5

In [30]:
# 面積列を変換
train['面積'] = train['面積'].apply(categorize_access)
test['面積'] = test['面積'].apply(categorize_access)

In [31]:
# 所在階列の「階」と「階建」を取り出して計算する関数
def calc_floor_ratio(floor_info):
    try:
        # 「階」と「階建」を分割
        floor, total_floors = floor_info.split('階／')
        # 数値に変換して割り算
        return int(floor) / int(total_floors.replace('階建', ''))
    except:
        return None  # エラー時はNoneを返す

In [32]:
# 所在階を変換
train['所在階'] = train['所在階'].apply(calc_floor_ratio)
test['所在階'] = test['所在階'].apply(calc_floor_ratio)

In [33]:
# 欠損値の補完（平均値で補完）
#mean_value = train['所在階'].mean()  # 平均値を計算
#train['所在階'] = train['所在階'].fillna(mean_value)  # 平均値で欠損値を補完
#mean_value2 = test['所在階'].mean()  # 平均値を計算
#test['所在階'] = test['所在階'].fillna(mean_value2)  # 平均値で欠損値を補完

In [34]:
# 最上階フラグ列を作成
train['最上階フラグ'] = train['所在階'].apply(lambda x: 1 if x == 1.0 else 0)
test['最上階フラグ'] = test['所在階'].apply(lambda x: 1 if x == 1.0 else 0)

In [35]:
# 建物構造を数値に変換するマッピング
location_mapping3 = {location: idx for idx, location in enumerate(train['建物構造'].unique())}

In [36]:
# 建物構造を数値に変換するマッピング(手動)
location_mapping3 = {
    'RC（鉄筋コンクリート）': 0,
    'SRC（鉄骨鉄筋コンクリート）': 0,  # コンクリート系にまとめる
    '木造': 1,
    '鉄骨造': 2,
    '軽量鉄骨': 3,  # 鉄骨造にまとめる
    'ALC（軽量気泡コンクリート）': 0,  # コンクリート系にまとめる
    'その他': 4,
    'PC（プレキャスト・コンクリート（鉄筋コンクリート））': 0,  # コンクリート系にまとめる
    'HPC（プレキャスト・コンクリート（重量鉄骨））': 0,  # コンクリート系にまとめる
    'ブロック': 0  # コンクリート系にまとめる
}

In [37]:
# 建物構造を数値に変換
train['建物構造'] = train['建物構造'].map(location_mapping3)
test['建物構造'] = test['建物構造'].map(location_mapping3)

In [38]:
# 不要な列を削除
train.drop(['id', '所在地', '所在階', '方角', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '契約期間'], axis=1, inplace=True)
test.drop(['id', '所在地', '所在階', '方角', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '契約期間'], axis=1, inplace=True)

In [39]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31470 entries, 0 to 31469
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   賃料      31470 non-null  int64  
 1   アクセス    31470 non-null  int64  
 2   間取り     31470 non-null  int64  
 3   築年数     31470 non-null  int64  
 4   面積      31470 non-null  int64  
 5   建物構造    31470 non-null  int64  
 6   緯度      31470 non-null  float64
 7   経度      31470 non-null  float64
 8   最上階フラグ  31470 non-null  int64  
dtypes: float64(2), int64(7)
memory usage: 2.2 MB


In [40]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31262 entries, 0 to 31261
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   アクセス    31262 non-null  int64  
 1   間取り     31256 non-null  float64
 2   築年数     31262 non-null  int64  
 3   面積      31262 non-null  int64  
 4   建物構造    31259 non-null  float64
 5   緯度      31262 non-null  float64
 6   経度      31262 non-null  float64
 7   最上階フラグ  31262 non-null  int64  
dtypes: float64(4), int64(4)
memory usage: 1.9 MB


In [41]:
train.head()

,賃料,アクセス,間取り,築年数,面積,建物構造,緯度,経度,最上階フラグ
0,75000,1,0,1,3,0,35.758248,139.729155,0
1,76000,1,1,5,2,2,35.662900,139.779212,0
2,110000,1,0,1,3,0,35.675147,139.667317,0
3,150000,1,2,3,6,0,35.700074,139.651613,0
4,74000,1,3,4,4,1,35.764544,139.869137,0


In [42]:
# plt.rcParams['font.family'] = 'IPAexGothic'
# sns.pairplot(train)

In [43]:
#plt.hist(train['間取り'])

In [44]:
# 最大値を確認
#max_num = train['間取り'].max()
#print(f"最大値: {max_num}")

In [45]:
# 分布確認
#print("間取りの分布:")
#print(train['間取り'].value_counts())

In [46]:
#plt.hist(train['建物構造'])

In [47]:
# 分布確認
#print("建物構造の分布:")
#print(train['建物構造'].value_counts())

In [48]:
#plt.hist(train['所在地'])

In [49]:
# 分布確認
#print("所在地の分布:")
#print(train['所在地'].value_counts())

In [50]:
#plt.hist(train['アクセス'])

In [51]:
# 分布確認
#print("アクセスの分布:")
#print(train['アクセス'].value_counts())

In [52]:
#plt.hist(train['築年数'])

In [53]:
# 最大値を確認
#max_num = train['築年数'].max()
#print(f"最大値: {max_num}")

In [54]:
# 分布確認
#print("築年数の分布:")
#print(train['築年数'].value_counts())

plt.hist(train['面積'])

# 最大値を確認
max_num = train['面積'].max()
print(f"最大値: {max_num}")

# 分布確認
print("面積の分布:")
print(train['面積'].value_counts())

In [55]:
#plt.hist(train['所在階'])

In [56]:
# 最大値を確認
#max_num = train['所在階'].max()
#print(f"最大値: {max_num}")

In [57]:
# 分布確認
#print("所在階の分布:")
#print(train['所在階'].value_counts())

In [58]:
train.to_csv('train_pre.csv', index=False)
test.to_csv('test_pre.csv', index=False)